# Stage 20 — Graph features & correlation (with MP denoising)

**Owner:** Data Scientist · graph  
**Pipeline stage:** `pipelines.stage_20_graph`

Build the transaction graph and derive the borrower-borrower correlation matrix the copula consumes. **arpym concept #2 lives here**: set `denoise=True` to apply Marčenko-Pastur spectrum shrinkage before the PSD projection — better conditioning and out-of-sample stability.

**Reads:** `persons_scored`, `transactions`  
**Writes:** `corr_matrix`, `network_stats`, `spectrum_diagnostics` (when denoising)

> This notebook is *modular*: it only needs its upstream artifacts to exist in the
> shared store. You can re-run just this stage without touching the others.


## 1. Setup

In [ ]:
# --- setup: make the project root importable + open the shared artifact store ---
import sys, os
ROOT = os.path.abspath("..")          # notebooks/ lives one level below the project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pipelines import ArtifactStore
store = ArtifactStore("output/etl")   # the SAME store every stage reads/writes
print("artifact store:", store.root)
print("existing artifacts:", store.list())


## 2. Run this stage

In [ ]:
from pipelines.stage_20_graph import run as run_graph

# Toggle denoise to compare. With denoising, spectrum_diagnostics records the
# conditioning improvement vs the raw matrix.
result = run_graph(store, denoise=True, denoise_method="mp_edge")
print(result.summary())


## 3. Inspect what this stage produced

In [ ]:
import numpy as np
corr = store.read_array("corr_matrix")
print("corr shape:", corr.shape, "| PSD:", np.linalg.eigvalsh(corr).min() > -1e-9)
print("network_stats:", store.read_json("network_stats"))
if store.exists("spectrum_diagnostics"):
    print("spectrum_diagnostics:", store.read_json("spectrum_diagnostics"))
